In [395]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from faker import Faker
import random

# Random seed for reproducibility
random.seed(42)
np.random.seed(42)


#### Clients


In [396]:
# Generate 200 unique clients
num_clients = 200
client_ids = [f'C{1000 + i}' for i in range(num_clients)]


# Possible values
symbols = ['EURUSD','USDJPY','GBPUSD','USDCHF', 'AUDUSD','USDCAD','NZDUSD','EURGBP','EURJPY','GBPJPY', 'BTCUSD']
account_types = ['Standard', 'ECN', 'Micro']
countries = ['MY', 'PH', 'SG', 'TH', 'VN', 'ID']
statuses = ['Approved', 'Rejected', 'Pending']
leverage_options = ['1:100', '1:200', '1:500', '1:1000']
spread_ranges = {
    'EURUSD': (0.8, 1.5),
    'USDJPY': (1.0, 2.0),
    'GBPUSD': (0.7,1.4),
    'USDCHF': (2.0, 5.0), 
    'AUDUSD': (1.0, 2.5),       
    'USDCAD': (1.5, 3.0),
    'NZDUSD': (1.2, 2.8),
    'EURGBP': (0.9, 1.8),
    'EURJPY': (1.0, 2.2),
    'GBPJPY': (1.5, 3.5),
    'BTCUSD': (10.0, 20.0),
}

# generate csv clients
clients = pd.DataFrame({
    'client_id': client_ids,
    'account_type': np.random.choice(account_types, num_clients),
    'country': np.random.choice(countries, num_clients),
    'signup_date': [datetime(2024, 1, 1) + timedelta(days=random.randint(0, 364)) for _ in range(num_clients)],
    'is_active': np.random.choice([True, False], num_clients, p=[0.76, 0.24])
})

clients.head()

,client_id,account_type,country,signup_date,is_active
0,C1000,Micro,TH,2024-11-23,True
1,C1001,Standard,MY,2024-02-27,True
2,C1002,Micro,ID,2024-01-13,False
3,C1003,Micro,MY,2024-05-20,True
4,C1004,Standard,PH,2024-05-05,True


#### Trades

In [397]:
num_trades = 2121
trade_ids = [f'T{1000 + i}' for i in range(num_trades)]

trades = []
for i in range(num_trades):
    client = np.random.choice(client_ids)
    symbol = np.random.choice(symbols)
    open_time = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 180), minutes=random.randint(0, 1440))
    volume = round(np.random.uniform(0.1, 5.0), 2)
    profit = round(np.random.normal(0, 50), 2)
    leverage = np.random.choice(leverage_options)
    spread = round(np.random.uniform(*spread_ranges[symbol]), 2)

    # 10% of trades will be left open (no close_time)
    if random.random() < 0.1:
        close_time = pd.NaT
    # 20% of trades will last 1–5 days
    elif random.random() < 0.2:
        close_time = open_time + timedelta(days=random.randint(1, 5))
    # 70% of trades will last a few hours
    else:
        close_time = open_time + timedelta(minutes=random.randint(1, 320))

    trades.append([
        trade_ids[i], client, symbol,
        open_time, close_time, volume,
        profit, leverage, spread
    ])

# Create DataFrame
trades_df = pd.DataFrame(trades, columns=[
    'trade_id', 'client_id', 'symbol', 'open_time', 'close_time',
    'volume', 'profit', 'leverage', 'spread'
])

# Preview
print(trades_df.head())

  trade_id client_id  symbol           open_time          close_time  volume  \
0    T1000     C1117  AUDUSD 2024-05-20 05:38:00 2024-05-20 09:15:00    4.97   
1    T1001     C1046  EURJPY 2024-02-24 18:24:00 2024-02-24 21:04:00    1.16   
2    T1002     C1116  USDCAD 2024-04-12 22:55:00 2024-04-13 03:20:00    2.96   
3    T1003     C1121  USDCHF 2024-04-25 04:07:00 2024-04-26 04:07:00    2.37   
4    T1004     C1027  AUDUSD 2024-05-30 18:54:00 2024-05-30 19:31:00    4.87   

   profit leverage  spread  
0    4.88   1:1000    1.11  
1  -38.65    1:500    1.55  
2   -5.55   1:1000    2.00  
3   48.82    1:200    2.69  
4  -95.64   1:1000    1.90  


In [398]:
trades_df.dtypes

trade_id              object
client_id             object
symbol                object
open_time     datetime64[ns]
close_time    datetime64[ns]
volume               float64
profit               float64
leverage              object
spread               float64
dtype: object

#### Transactions 


In [399]:
# Generate Transaction data
num_transactions = 600
transaction_ids = [f'TX{1000 + i}' for i in range(num_transactions)]

transactions = []
for i in range(num_transactions):
    client = np.random.choice(client_ids)
    tx_type = np.random.choice(['Deposit', 'Withdrawal'])
    amount = round(np.random.uniform(50, 10000), 2)
    tx_date = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 180))
    status = np.random.choice(statuses, p=[0.7, 0.2, 0.1])
    transactions.append([transaction_ids[i], client, tx_type, amount, tx_date, status])
    
transactions_df = pd.DataFrame(transactions, columns=['transaction_id', 'client_id', 'tx_type', 'amount', 'tx_date', 'status'])
transactions_df.head()

,transaction_id,client_id,tx_type,amount,tx_date,status
0,TX1000,C1094,Deposit,6140.19,2024-04-01,Rejected
1,TX1001,C1142,Withdrawal,6895.44,2024-02-26,Approved
2,TX1002,C1108,Withdrawal,1515.37,2024-03-12,Approved
3,TX1003,C1017,Deposit,4240.85,2024-02-15,Rejected
4,TX1004,C1036,Withdrawal,1959.17,2024-03-06,Approved


In [400]:
transactions_df.dtypes

transaction_id            object
client_id                 object
tx_type                   object
amount                   float64
tx_date           datetime64[ns]
status                    object
dtype: object

#### Account Snapshots

In [401]:
# set date range for account snapshots

# convert dates
trades_df['open_time'] = pd.to_datetime(trades_df['open_time'])
trades_df['close_time'] = pd.to_datetime(trades_df['close_time'])
transactions_df['tx_date'] = pd.to_datetime(transactions_df['tx_date'])

date_range = pd.date_range(start='2024-06-01', end='2024-06-30')

snapshots = []
for date in date_range:
    for client_id in clients['client_id']:
        dep = transactions_df.query("client_id == @client_id and tx_type == 'Deposit' and tx_date <= @date")['amount'].sum()
        wd = transactions_df.query("client_id == @client_id and tx_type == 'Withdrawal' and tx_date <= @date")['amount'].sum()
        closed_pnl = trades_df.query("client_id == @client_id and close_time <= @date")['profit'].sum()
        
        balance = dep - wd + closed_pnl
        
        open_trades = trades_df.query("client_id == @client_id and open_time <= @date and (close_time > @date or close_time.isna())")
        floating_pnl = open_trades['profit'].sum()
        swap = round(np.random.uniform(-10, 5), 2) if not open_trades.empty else 0
        margin_used = round(open_trades['volume'].sum() * 1000 / 100, 2)
        equity = balance + floating_pnl + swap
        free_margin = equity - margin_used
        
        snapshots.append({
            'client_id': client_id,
            'date': date.date(),
            'balance': balance,
            'equity': equity,
            'floating_pnl': floating_pnl,
            'swap': swap,
            'margin_used': margin_used,
            'free_margin': free_margin
        })
        
snapshots_df = pd.DataFrame(snapshots)
snapshots_df.head()
        

,client_id,date,balance,equity,floating_pnl,swap,margin_used,free_margin
0,C1000,2024-06-01,-303.13,-303.13,0.00,0.00,0.0,-303.13
1,C1001,2024-06-01,58.15,58.15,0.00,0.00,0.0,58.15
2,C1002,2024-06-01,-30.86,-30.86,0.00,0.00,0.0,-30.86
3,C1003,2024-06-01,-10145.61,-10123.15,27.17,-4.71,9.1,-10132.25
4,C1004,2024-06-01,9766.78,9818.30,55.89,-4.37,50.2,9768.10


In [402]:
snapshots_df.iloc[5999]

client_id            C1199
date            2024-06-30
balance            8174.57
equity             8266.05
floating_pnl        100.95
swap                 -9.47
margin_used           20.1
free_margin        8245.95
Name: 5999, dtype: object

#### Client Cashflow

In [403]:
# Generate Client Cashflow
cashflow_df = transactions_df.copy()

cashflow_df['description'] = cashflow_df['tx_type'].map({
    'Deposit': 'Deposit Made',
    'Withdrawal': 'Withdrawal requested'
})

cashflow_df.head()

,transaction_id,client_id,tx_type,amount,tx_date,status,description
0,TX1000,C1094,Deposit,6140.19,2024-04-01,Rejected,Deposit Made
1,TX1001,C1142,Withdrawal,6895.44,2024-02-26,Approved,Withdrawal requested
2,TX1002,C1108,Withdrawal,1515.37,2024-03-12,Approved,Withdrawal requested
3,TX1003,C1017,Deposit,4240.85,2024-02-15,Rejected,Deposit Made
4,TX1004,C1036,Withdrawal,1959.17,2024-03-06,Approved,Withdrawal requested


#### Trade Analysis

In [404]:
analysis = []

for _, row in trades_df.iterrows():
    duration = (row['close_time'] - row['open_time']).total_seconds() / 60  # minutes
    rr = round(abs(row['profit'] / 10), 2)
    strategy = 'Scalping' if duration <= 30 else 'Day Trading' if duration <= 240 else 'Swing Trading'
    
    analysis.append({
        'trade_id': row['trade_id'],
        'client_id': row['client_id'],
        'symbol': row['symbol'],
        'entry': row['open_time'],
        'exit': row['close_time'],
        'pnl': row['profit'],
        'duration_min': duration,
        'risk_reward': rr,
        'strategy': strategy,
    })
    
trade_analysis_df = pd.DataFrame(analysis)
trade_analysis_df.head()

,trade_id,client_id,symbol,entry,exit,pnl,duration_min,risk_reward,strategy
0,T1000,C1117,AUDUSD,2024-05-20 05:38:00,2024-05-20 09:15:00,4.88,217.0,0.49,Day Trading
1,T1001,C1046,EURJPY,2024-02-24 18:24:00,2024-02-24 21:04:00,-38.65,160.0,3.86,Day Trading
2,T1002,C1116,USDCAD,2024-04-12 22:55:00,2024-04-13 03:20:00,-5.55,265.0,0.55,Swing Trading
3,T1003,C1121,USDCHF,2024-04-25 04:07:00,2024-04-26 04:07:00,48.82,1440.0,4.88,Swing Trading
4,T1004,C1027,AUDUSD,2024-05-30 18:54:00,2024-05-30 19:31:00,-95.64,37.0,9.56,Day Trading


In [405]:
trade_analysis_df['strategy'].value_counts()

strategy
Swing Trading    1003
Day Trading       984
Scalping          134
Name: count, dtype: int64

#### Client Risk Score


In [406]:
scores = []

for client_id in clients['client_id']:
    client_trades = trades_df[trades_df['client_id'] == client_id]
    snap = snapshots_df[snapshots_df['client_id'] == client_id]
    
    avg_dd = snap['floating_pnl'].mean()
    max_dd = snap['floating_pnl'].min()
    avg_margin = snap['margin_used'].mean()
    win_rate = (client_trades['profit'] > 0).mean()
    trade_count = len(client_trades)
    
    risk = (
        "High" if max_dd < -1000 or avg_margin > 700
        else "Medium" if win_rate < 0.5
        else "Low"
    )
    
    scores.append({
        'client_id': client_id,
        'avg_drawdown': avg_dd,
        'max_drawdown': max_dd,
        'avg_margin_used': avg_margin,
        'win_rate_pct': win_rate * 100,
        'trade_count': trade_count,
        'risk_score': risk
    })

client_risk_score_df = pd.DataFrame(scores)
client_risk_score_df.head()


,client_id,avg_drawdown,max_drawdown,avg_margin_used,win_rate_pct,trade_count,risk_score
0,C1000,0.000,0.00,0.00,27.272727,11,Medium
1,C1001,0.439,0.00,4.33,81.818182,11,Low
2,C1002,-92.515,-122.70,17.46,33.333333,12,Medium
3,C1003,27.170,27.17,9.10,66.666667,9,Low
4,C1004,55.890,55.89,50.20,66.666667,9,Low


### Data Checking & Transformation

#### 1. Clients DF

In [407]:
print(clients.head())
print(clients.dtypes)

  client_id account_type country signup_date  is_active
0     C1000        Micro      TH  2024-11-23       True
1     C1001     Standard      MY  2024-02-27       True
2     C1002        Micro      ID  2024-01-13      False
3     C1003        Micro      MY  2024-05-20       True
4     C1004     Standard      PH  2024-05-05       True
client_id               object
account_type            object
country                 object
signup_date     datetime64[ns]
is_active                 bool
dtype: object


In [408]:
# mapping original country names to their codes

clients['country'].value_counts()

country_mapping = {
    'MY': 'Malaysia',
    'SG': 'Singapore',
    'TH': 'Thailand',
    'ID': 'Indonesia',
    'PH': 'Philippines',
    'VN': 'Vietnam'
}
clients['country_name'] = clients['country'].map(country_mapping) 
clients.head()  

# mapping True/False - active / not active
clients['is_active'] =  clients['is_active'].map({True:'Active', False: 'Inactive'})
 

# renam and rearrange columns
clients = clients.rename(columns={
    'client_id':'client_id',
    'account_type':'account_type',
    'country':'country_code',
    'country_name':'country_name',
    'signup_date':'signup_date',
    'is_active':'account_status'
})

clients = clients[['client_id', 'account_type', 'signup_date', 'account_status','country_code', 'country_name']]
clients_df = clients.copy()
clients_df.head()


,client_id,account_type,signup_date,account_status,country_code,country_name
0,C1000,Micro,2024-11-23,Active,TH,Thailand
1,C1001,Standard,2024-02-27,Active,MY,Malaysia
2,C1002,Micro,2024-01-13,Inactive,ID,Indonesia
3,C1003,Micro,2024-05-20,Active,MY,Malaysia
4,C1004,Standard,2024-05-05,Active,PH,Philippines


#### Trades Df

In [409]:
print(trades_df.head())

  trade_id client_id  symbol           open_time          close_time  volume  \
0    T1000     C1117  AUDUSD 2024-05-20 05:38:00 2024-05-20 09:15:00    4.97   
1    T1001     C1046  EURJPY 2024-02-24 18:24:00 2024-02-24 21:04:00    1.16   
2    T1002     C1116  USDCAD 2024-04-12 22:55:00 2024-04-13 03:20:00    2.96   
3    T1003     C1121  USDCHF 2024-04-25 04:07:00 2024-04-26 04:07:00    2.37   
4    T1004     C1027  AUDUSD 2024-05-30 18:54:00 2024-05-30 19:31:00    4.87   

   profit leverage  spread  
0    4.88   1:1000    1.11  
1  -38.65    1:500    1.55  
2   -5.55   1:1000    2.00  
3   48.82    1:200    2.69  
4  -95.64   1:1000    1.90  


In [410]:

# handling nulls - replacing close time with open time + avg tradiing duration
avg_duration = round(((trades_df['close_time'] - trades_df['open_time']).dt.total_seconds() / 60).mean(), 0)
print(avg_duration)

trades_df['close_time']= trades_df['close_time'].fillna(
    trades_df['open_time'] + pd.to_timedelta(avg_duration, unit='m')
)

# new calculated columns
trades_df['trade_duration_min'] = (trades_df['close_time'] - trades_df['open_time']).dt.total_seconds() / 60
trades_df['trade_hour'] = trades_df['open_time'].dt.hour

# mapping hour to trade session
session_labels = []
for hour in trades_df['trade_hour']:
    if 6 <= hour < 14:
        session_labels.append('Sydney')
    elif 8 <= hour < 17:
        session_labels.append('Tokyo')
    elif hour >= 16 or hour < 1:
        session_labels.append('London')
    elif hour >= 21 or hour < 6:
        session_labels.append('New York')
    else:
        session_labels.append('Unknown')
        
trades_df['trade_session'] = session_labels






992.0


In [411]:
# rearrange columns
trades_df = trades_df[['trade_id', 'client_id', 'symbol', 'open_time', 'close_time', 'trade_duration_min', 'trade_hour', 'leverage', 'spread',  'volume', 'profit',  'trade_session']]
trades_df.head()

,trade_id,client_id,symbol,open_time,close_time,trade_duration_min,trade_hour,leverage,spread,volume,profit,trade_session
0,T1000,C1117,AUDUSD,2024-05-20 05:38:00,2024-05-20 09:15:00,217.0,5,1:1000,1.11,4.97,4.88,New York
1,T1001,C1046,EURJPY,2024-02-24 18:24:00,2024-02-24 21:04:00,160.0,18,1:500,1.55,1.16,-38.65,London
2,T1002,C1116,USDCAD,2024-04-12 22:55:00,2024-04-13 03:20:00,265.0,22,1:1000,2.00,2.96,-5.55,London
3,T1003,C1121,USDCHF,2024-04-25 04:07:00,2024-04-26 04:07:00,1440.0,4,1:200,2.69,2.37,48.82,New York
4,T1004,C1027,AUDUSD,2024-05-30 18:54:00,2024-05-30 19:31:00,37.0,18,1:1000,1.90,4.87,-95.64,London


#### Transaction Df

In [412]:
transactions_df.head()

,transaction_id,client_id,tx_type,amount,tx_date,status
0,TX1000,C1094,Deposit,6140.19,2024-04-01,Rejected
1,TX1001,C1142,Withdrawal,6895.44,2024-02-26,Approved
2,TX1002,C1108,Withdrawal,1515.37,2024-03-12,Approved
3,TX1003,C1017,Deposit,4240.85,2024-02-15,Rejected
4,TX1004,C1036,Withdrawal,1959.17,2024-03-06,Approved


In [413]:

# Rejected Reasons
deposit_rejected_reason = [
    'Mismatched account name',
    'Incomplete payment details',
    'Insufficient funds',
    'Technical errors or delays',
    'Unsupported payment method'
]

withdrawal_rejected_reason = [
    'Pending document approval',
    'Insufficient balance',
    'Suspicious or flagged activity'
]

transactions_df['rejected_reason'] = transactions_df.apply(
    lambda row: random.choice(deposit_rejected_reason) if row['tx_type'] == 'Deposit' and row['status'] == 'Rejected' else
                 random.choice(withdrawal_rejected_reason) if row['tx_type'] == 'Withdrawal' and row['status'] == 'Rejected' else
                 None, axis=1
)

transactions_df.head()

,transaction_id,client_id,tx_type,amount,tx_date,status,rejected_reason
0,TX1000,C1094,Deposit,6140.19,2024-04-01,Rejected,Unsupported payment method
1,TX1001,C1142,Withdrawal,6895.44,2024-02-26,Approved,None
2,TX1002,C1108,Withdrawal,1515.37,2024-03-12,Approved,None
3,TX1003,C1017,Deposit,4240.85,2024-02-15,Rejected,Incomplete payment details
4,TX1004,C1036,Withdrawal,1959.17,2024-03-06,Approved,None


#### Accounts Snapshots Df

In [414]:
snapshots_df.head()

,client_id,date,balance,equity,floating_pnl,swap,margin_used,free_margin
0,C1000,2024-06-01,-303.13,-303.13,0.00,0.00,0.0,-303.13
1,C1001,2024-06-01,58.15,58.15,0.00,0.00,0.0,58.15
2,C1002,2024-06-01,-30.86,-30.86,0.00,0.00,0.0,-30.86
3,C1003,2024-06-01,-10145.61,-10123.15,27.17,-4.71,9.1,-10132.25
4,C1004,2024-06-01,9766.78,9818.30,55.89,-4.37,50.2,9768.10


In [415]:
snapshots_df.columns

Index(['client_id', 'date', 'balance', 'equity', 'floating_pnl', 'swap',
       'margin_used', 'free_margin'],
      dtype='object')

In [416]:
# Rearrange columns
snapshots_df = snapshots_df[['client_id', 'date', 'floating_pnl', 'swap', 'margin_used', 'free_margin','balance', 'equity']]

# Mapping account status based on - balance, free margin, and equity
snapshots_df['account_status'] = snapshots_df.apply(
    lambda x: 'Negative Balance' if x['balance'] < 0 else
              'Margin Call' if x['free_margin'] < 0 else
              'Active', axis = 1
)

snapshots_df.head()

,client_id,date,floating_pnl,swap,margin_used,free_margin,balance,equity,account_status
0,C1000,2024-06-01,0.00,0.00,0.0,-303.13,-303.13,-303.13,Negative Balance
1,C1001,2024-06-01,0.00,0.00,0.0,58.15,58.15,58.15,Active
2,C1002,2024-06-01,0.00,0.00,0.0,-30.86,-30.86,-30.86,Negative Balance
3,C1003,2024-06-01,27.17,-4.71,9.1,-10132.25,-10145.61,-10123.15,Negative Balance
4,C1004,2024-06-01,55.89,-4.37,50.2,9768.10,9766.78,9818.30,Active


#### Cashflow Df

In [417]:
cashflow_df.head()

,transaction_id,client_id,tx_type,amount,tx_date,status,description
0,TX1000,C1094,Deposit,6140.19,2024-04-01,Rejected,Deposit Made
1,TX1001,C1142,Withdrawal,6895.44,2024-02-26,Approved,Withdrawal requested
2,TX1002,C1108,Withdrawal,1515.37,2024-03-12,Approved,Withdrawal requested
3,TX1003,C1017,Deposit,4240.85,2024-02-15,Rejected,Deposit Made
4,TX1004,C1036,Withdrawal,1959.17,2024-03-06,Approved,Withdrawal requested


#### Trade Analysis Df 

In [418]:
trade_analysis_df.head()

,trade_id,client_id,symbol,entry,exit,pnl,duration_min,risk_reward,strategy
0,T1000,C1117,AUDUSD,2024-05-20 05:38:00,2024-05-20 09:15:00,4.88,217.0,0.49,Day Trading
1,T1001,C1046,EURJPY,2024-02-24 18:24:00,2024-02-24 21:04:00,-38.65,160.0,3.86,Day Trading
2,T1002,C1116,USDCAD,2024-04-12 22:55:00,2024-04-13 03:20:00,-5.55,265.0,0.55,Swing Trading
3,T1003,C1121,USDCHF,2024-04-25 04:07:00,2024-04-26 04:07:00,48.82,1440.0,4.88,Swing Trading
4,T1004,C1027,AUDUSD,2024-05-30 18:54:00,2024-05-30 19:31:00,-95.64,37.0,9.56,Day Trading


#### Client Risk Score Df

In [419]:
client_risk_score_df.head()

,client_id,avg_drawdown,max_drawdown,avg_margin_used,win_rate_pct,trade_count,risk_score
0,C1000,0.000,0.00,0.00,27.272727,11,Medium
1,C1001,0.439,0.00,4.33,81.818182,11,Low
2,C1002,-92.515,-122.70,17.46,33.333333,12,Medium
3,C1003,27.170,27.17,9.10,66.666667,9,Low
4,C1004,55.890,55.89,50.20,66.666667,9,Low


#### Final Checking

In [420]:
print(clients_df.info())
print(trades_df.info())
print(transactions_df.info())
print(snapshots_df.info())
print(cashflow_df.info())
print(trade_analysis_df.info())
print(client_risk_score_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   client_id       200 non-null    object        
 1   account_type    200 non-null    object        
 2   signup_date     200 non-null    datetime64[ns]
 3   account_status  200 non-null    object        
 4   country_code    200 non-null    object        
 5   country_name    200 non-null    object        
dtypes: datetime64[ns](1), object(5)
memory usage: 9.5+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2121 entries, 0 to 2120
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   trade_id            2121 non-null   object        
 1   client_id           2121 non-null   object        
 2   symbol              2121 non-null   object        
 3   open_time           2121

### Saving Output




In [421]:
path_to_save = 'C:/Users/User/Desktop/Data Analyst/End To End Project/Trading/'
file_list = [clients_df, trades_df, transactions_df, snapshots_df, cashflow_df, trade_analysis_df, client_risk_score_df]
file_names = ['clients_df', 'trades_df', 'transactions_df', 'snapshots_df', 'cashflow_df', 'trade_analysis_df', 'client_risk_score_df']

# functions to save files
def save_df(df, file_path, file_name):
    df.to_csv(f"{file_path}{file_name}.csv", index=False)
    
for name, df in zip(file_names, file_list):
    save_df(df, path_to_save, name)
    print(f"Saved {name} to {path_to_save}")
    

Saved clients_df to C:/Users/User/Desktop/Data Analyst/End To End Project/Trading/
Saved trades_df to C:/Users/User/Desktop/Data Analyst/End To End Project/Trading/
Saved transactions_df to C:/Users/User/Desktop/Data Analyst/End To End Project/Trading/
Saved snapshots_df to C:/Users/User/Desktop/Data Analyst/End To End Project/Trading/
Saved cashflow_df to C:/Users/User/Desktop/Data Analyst/End To End Project/Trading/
Saved trade_analysis_df to C:/Users/User/Desktop/Data Analyst/End To End Project/Trading/
Saved client_risk_score_df to C:/Users/User/Desktop/Data Analyst/End To End Project/Trading/
